In [4]:
import os
import sys
sys.path.append(r"C:\Users\Shisir\Desktop\Physics_LLM_Project\Problems\Text")


from text_prompts import regular_system_prompts 
from text_prompts import detail_system_prompts

from dotenv import load_dotenv
from huggingface_hub import login

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [6]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

## authenticate with huggingface

login(token=hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
## check if cuda ia available and GPU

print("CUDA available:", torch.cuda.is_available())
print("Using GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Using GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [8]:
## select model id
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [9]:
## download and load the tokenizer
## we will use automodelforcausal for more options

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
).to("cuda")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [10]:
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [11]:
print(regular_system_prompts) 

{'Problem_1_Regular_Q': 'You are a physics expert. Based on the question that is given solve the problem.', 'Problem_1_Obvious_Q': 'You are a physics expert. Based on the question that is given solve the problem.', 'Problem_1_non_Obvious_Q': 'You are a physics expert. Based on the question that is given solve the problem.', 'Problem_2_Regular_Q': 'You are a physics expert. Based on the question that is given solve the problem.', 'Problem_2_Obvious_Q': 'You are a physics expert. Based on the question that is given solve the problem.', 'Problem_2_non_obvious_Q': 'You are a physics expert. Based on the question that is given solve the problem.', 'Problem_3_Regular_Q': 'You are a physics expert. Based on the question that is given solve the problem.', 'Problem_3_Obvious_Q': 'You are a physics expert. Based on the question that is given solve the problem.', 'Problem_3_non_obvious_Q': 'You are a physics expert. Based on the question that is given solve the problem.'}


In [14]:
sys_prompt =(regular_system_prompts["Problem_1_Regular_Q"])
sys_prompt

'You are a physics expert. Based on the question that is given solve the problem.'

In [15]:
PATH=os.path.join('..','..','Problems','Text','Problem_1_Regular_Q.md')
with open(PATH, "r", encoding="utf-8") as f:
    Problem_1_Regular = f.read()
print(Problem_1_Regular)
print(type(Problem_1_Regular))

A block of mass (m) = 12 kg slides down from rest in an inclined plane. The plane is at a constant angle ($\theta$) of 30$^o$ with respect to the horizontal. The coefficient of kinetic friction ($\mu_k$) between the mass and the plane is 0.5. The acceleration due to gravity in the location is 10 m/s$^2$. The block slides down a total distance of 5 meters before it touches the ground. What is the total time taken by the block to travel the 5 meters? 
<class 'str'>


In [16]:
messages = [
    {"role": "system", "content": sys_prompt},
    {"role": "user", "content": Problem_1_Regular},
]

In [17]:
# Format the messages, tokenize them, and move them to the GPU.

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

In [18]:
# Generate the response without computing training gradients.
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

In [19]:
# Decode the generated response

new_tokens = outputs[:, inputs["input_ids"].shape[1]:]

In [21]:
response = tokenizer.decode(new_tokens, skip_special_tokens=True)
print("\nResponse:\n", response)


Response:
 ["Based on the given information, the block of mass (m) = 12 kg, coefficient of kinetic friction ($\\mu_k$) = 0.5, and acceleration due to gravity in the location (m/s$^2$) = 10 m/s$^2$ in the location of the block's sliding, the block slides down a total distance of 5 meters before it touches the ground.\n\nThe total time taken by the block to travel the 5 meters is the product of the distance traveled and the time it takes to travel that distance. In this case, the total distance traveled is 5 meters, which is given by:\n\nDistance traveled = 5 m\nTime taken = (5 m) / (10 m/s^2) = 5 s\n\nTherefore, the total time taken by the block to travel the 5 meters is 5 s."]
